# ABSA Pipeline (Kaggle GPU Ready)

This notebook is designed for Kaggle GPU and local runs.

Pipeline:
- data cleaning + label normalization
- leakage-safe split (group by normalized text)
- aspect inference from pretrained model (no aspect training)
- class balancing (train-only) + sentiment training
- save metrics, figures, and business insights artifacts

Best-practice notes (Context7):
- Resampling must happen inside train split only.
- Track macro-F1/per-class metrics for imbalanced multiclass.
- For Trainer: use `eval_strategy`, `load_best_model_at_end`, and mixed precision (`fp16`/`bf16`) on CUDA.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

IN_KAGGLE = Path('/kaggle').exists()
print('IN_KAGGLE =', IN_KAGGLE)

if IN_KAGGLE:
    !nvidia-smi || true
    WORKDIR = Path('/kaggle/working/final')
    WORKDIR.mkdir(parents=True, exist_ok=True)

    # Copy required project files from Kaggle input dataset to writable working dir.
    required = {'absa_pipeline.py', 'business_insights.py', 'requirements.txt', 'all_reviews_merged.csv'}
    found = {}
    for p in Path('/kaggle/input').rglob('*'):
        if p.name in required and p.name not in found:
            found[p.name] = p
    missing = required - set(found)
    if missing:
        raise FileNotFoundError(f'Missing required files in /kaggle/input: {sorted(missing)}')
    for name, src in found.items():
        shutil.copy2(src, WORKDIR / name)

    os.chdir(WORKDIR)
    print('Working directory:', Path.cwd())
else:
    WORKDIR = Path.cwd()
    print('Working directory:', WORKDIR)

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import absa_pipeline as absa

CSV_PATH = Path('all_reviews_merged.csv')
ART = Path('artifacts')
ART.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1) Data cleaning

In [ ]:
df_raw = absa.load_raw(CSV_PATH)
df = absa.clean_labels(df_raw)

print('raw rows:', len(df_raw))
print('clean rows:', len(df))
print('\nSentiment distribution:')
print(df['sentiment'].value_counts())
print('\nAspect distribution (top 12):')
print(df['aspect'].value_counts().head(12))

## 2) Leakage-safe split

In [ ]:
split = absa.make_splits(df, n_splits=5, random_state=42)
absa.verify_no_group_leakage(df, split)

df['split'] = np.where(
    df['text_id'].isin(split.test_ids),
    'test',
    np.where(df['text_id'].isin(split.val_ids), 'val', 'train')
)
print(df['split'].value_counts())

## 3) Run training pipeline (FULL DATA by default)

- `--use_predicted_aspect`: aspect comes from pretrained zero-shot model
- `--train_transformer`: trains transformer sentiment model with balancing + macro-F1 selection
- `MAX_ROWS = None` means **train on all cleaned rows**
- For quick debug only, set `MAX_ROWS = 1000` (or lower)

In [ ]:
import argparse

# IMPORTANT:
# - None  => full dataset training
# - int   => quick debug subset (e.g., 1000)
MAX_ROWS = None

args = argparse.Namespace(
    csv='all_reviews_merged.csv',
    max_rows=MAX_ROWS,
    use_predicted_aspect=True,
    train_transformer=True,
    transformer_model='distilbert-base-uncased',
    epochs=3.0,
    batch_size=8,
    lr=2e-5,
)

absa.run(args)

## 4) Load results + save evaluation figures

In [ ]:
res_path = ART / 'results.json'
results = json.loads(res_path.read_text(encoding='utf-8'))
results['data']

In [ ]:
def plot_cm(cm, labels, title, out_path):
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.show()
    plt.close()

labels = results['baseline_tfidf_logreg']['test']['labels']
plot_cm(results['baseline_tfidf_logreg']['test']['confusion_matrix'], labels,
        'Baseline Test Confusion Matrix', ART / 'cm_baseline_test.png')

if 'transformer_sentiment' in results:
    plot_cm(results['transformer_sentiment']['test']['confusion_matrix'], labels,
            'Transformer Test Confusion Matrix', ART / 'cm_transformer_test.png')

In [ ]:
summary = {
    'baseline_variant': results['baseline_tfidf_logreg'].get('variant_selected'),
    'baseline_macro_f1': results['baseline_tfidf_logreg']['test']['report']['macro avg']['f1-score'],
    'baseline_accuracy': results['baseline_tfidf_logreg']['test']['report']['accuracy'],
}
if 'transformer_sentiment' in results:
    summary['transformer_macro_f1'] = results['transformer_sentiment']['test']['report']['macro avg']['f1-score']
    summary['transformer_accuracy'] = results['transformer_sentiment']['test']['report']['accuracy']
    summary['transformer_argmax_test_macro_f1'] = results['transformer_sentiment'].get('argmax_test_macro_f1')

(ART / 'results_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary

## 5) Business insights outputs

In [ ]:
import argparse
import business_insights as bi

bi.main(argparse.Namespace(csv='all_reviews_merged.csv'))

print('Saved artifacts in:', ART.resolve())
for p in [
    'results.json',
    'results_summary.json',
    'cm_baseline_test.png',
    'cm_transformer_test.png',
    'nss_overall.txt',
    'nss_daily.csv',
    'nss_by_aspect.csv',
    'nss_trend.png',
    'nss_by_aspect.png',
]:
    fp = ART / p
    print('-', fp, '| exists:', fp.exists())